In [ ]:
all_datasets = ['real_data/data_batteries_ecfp_descriptor', 'herg_data/data_herg_ecfp', 'synthetic_data/qm9_simple_linear6', 'synthetic_data/qm9_piecewise_linear_6','synthetic_data/qm9_nonlinear_6']
dataset_names = ['COF', 'hERG', 'QM9 Simple Linear', 'QM9 Piecewise Linear', 'QM9 Nonlinear']
#all_datasets = ['gt_synthetic_data/qm9_simple_linear6', 'gt_synthetic_data/qm9_piecewise_linear_6','gt_synthetic_data/qm9_nonlinear_6']
#dataset_names = [ 'QM9 Simple Linear', 'QM9 Piecewise Linear', 'QM9 Nonlinear']
metrics = {}
cf_validity = {}
cf_similarity = {}

import pandas as pd
import pickle

for i, dataset in enumerate(all_datasets):
    with open(f'../results/{dataset}/explanations/analysis/metrics_results.pickle', 'rb') as f:
        results = pickle.load(f)
    metrics[dataset_names[i]] = results
    with open(f'../results/{dataset}/explanations/analysis/cf_validity_results.pickle', 'rb') as f:
        results = pickle.load(f)
    cf_validity[dataset_names[i]] = results
    with open(f'../results/{dataset}/explanations/analysis/cf_similarity_results.pickle', 'rb') as f:
        results = pickle.load(f)
    cf_similarity[dataset_names[i]] = results

metrics

In [ ]:
import numpy as np
for i, dataset in enumerate(all_datasets):
    with open(f'../results/{dataset}/results.pickle', 'rb') as f:
        results = pickle.load(f)
    m_mean = {}
    m_std = {}
    for method in results['scores']:
        m_mean[method] = [np.mean(results['scores'][method])]
        m_std[method] = [np.std(results['scores'][method])]
    print(dataset)
    print('Mean:')
    display(pd.DataFrame(m_mean))
    print('Std:')
    display(pd.DataFrame(m_std))
    print('--'*20)


In [ ]:
def reshape_results_to_wide_format(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transforms a long-format DataFrame of results into a wide-format DataFrame.

    The resulting DataFrame will have 'method' as the index, and a single column
    for each combination of 'dataset' and 'metric'.

    Args:
        df (pd.DataFrame): A DataFrame with columns ['method', 'dataset', 'metric', 'value'].

    Returns:
        pd.DataFrame: The reshaped, wide-format DataFrame.
    """
    pivoted_df = df.pivot_table(
        index='method',
        columns=['dataset', 'metric'],
        values='value'
    )

    pivoted_df.columns = [f"{col[0]}_{col[1]}" for col in pivoted_df.columns]
    return pivoted_df

In [ ]:
cf_validity

In [ ]:
cf_similarity

In [ ]:
df_validity = {'method': [], 'dataset': [], 'metric': [], 'value': []}
for k in cf_validity:
    for method in cf_validity[k]:
        df_validity['method'].append(method)
        df_validity['dataset'].append(k)
        df_validity['metric'].append('validity')
        df_validity['value'].append(cf_validity[k][method])
df_validity = pd.DataFrame(df_validity)
df_validity = reshape_results_to_wide_format(df_validity)
df_validity = df_validity[[
    'QM9 Simple Linear_validity',
    'QM9 Piecewise Linear_validity',
    'QM9 Nonlinear_validity',
    'COF_validity',
    'hERG_validity',
]]
df_validity = df_validity.rename(
    columns={
        'QM9 Simple Linear_validity': 'QM9 Simple Linear',
        'QM9 Piecewise Linear_validity': 'QM9 Piecewise Linear',
        'QM9 Nonlinear_validity': 'QM9 Nonlinear',
        'COF_validity': 'COF',
        'hERG_validity': 'hERG',
    }
)
df_validity = df_validity.T
df_validity = df_validity[['mmace', 'meg']]
print(df_validity.style.format(precision=2).to_latex())

In [ ]:
df_validity = {'method': [], 'dataset': [], 'metric': [], 'value': []}
for k in cf_similarity:
    for method in cf_similarity[k]:
        df_validity['method'].append(method)
        df_validity['dataset'].append(k)
        df_validity['method'].append(method)
        df_validity['dataset'].append(k)
        df_validity['metric'].append('similarity')
        df_validity['metric'].append('similarity std')
        df_validity['value'].append(cf_similarity[k][method][0])
        df_validity['value'].append(cf_similarity[k][method][1])
df_validity = pd.DataFrame(df_validity)
df_validity = reshape_results_to_wide_format(df_validity)
df_validity_mean = df_validity[[
    'QM9 Simple Linear_similarity',
    'QM9 Piecewise Linear_similarity',
    'QM9 Nonlinear_similarity',
    'COF_similarity',
    'hERG_similarity'
]]
df_validity_mean = df_validity_mean.rename(
    columns={
        'QM9 Simple Linear_similarity': 'QM9 Simple Linear',
        'QM9 Piecewise Linear_similarity': 'QM9 Piecewise Linear',
        'QM9 Nonlinear_similarity': 'QM9 Nonlinear',
        'COF_similarity': 'COF',
        'hERG_similarity': 'hERG',
    }
)
df_validity_std = df_validity[[
    'QM9 Simple Linear_similarity std',
    'QM9 Piecewise Linear_similarity std',
    'QM9 Nonlinear_similarity std',
    'COF_similarity std',
    'hERG_similarity std'
]]
df_validity_std = df_validity_std.rename(
    columns={
        'QM9 Simple Linear_similarity std': 'QM9 Simple Linear',
        'QM9 Piecewise Linear_similarity std': 'QM9 Piecewise Linear',
        'QM9 Nonlinear_similarity std': 'QM9 Nonlinear',
        'COF_similarity std': 'COF',
        'hERG_similarity std': 'hERG',
    }
)
df_mean_str = df_validity_mean.map('{:.2f}'.format)
df_std_str = df_validity_std.map('{:.2f}'.format)
df_validity = df_mean_str + '(' + df_std_str + ')'
df_validity = df_validity.T
df_validity = df_validity[['mmace', 'meg']]
print(df_validity.style.format(precision=2).to_latex())

In [ ]:

metrics_dataframe = {'method': [], 'dataset': [], 'metric': [], 'value': []}
cf_validity_dataframe = {'method': [], 'dataset': [], 'metric': []}
cf_similarity_dataframe = {'method': [], 'dataset': [], 'metric': []}

for dataset in dataset_names:
    for method, method_results in metrics[dataset].items():

        #metrics_dataframe['method'].append(method)
        metrics_dataframe['dataset'].append(dataset)
        metrics_dataframe['method'].append(method)
        metrics_dataframe['dataset'].append(dataset)
        metrics_dataframe['method'].append(method)
        metrics_dataframe['dataset'].append(dataset)
        metrics_dataframe['method'].append(method)
        metrics_dataframe['dataset'].append(dataset)
        metrics_dataframe['method'].append(method)
        metrics_dataframe['metric'].append('pgi')
        metrics_dataframe['value'].append(method_results['pgi_mean'])
        metrics_dataframe['metric'].append('pgi std')
        metrics_dataframe['value'].append(method_results['pgi_std'])
        metrics_dataframe['metric'].append('pgu')
        metrics_dataframe['value'].append(method_results['pgu_mean'])
        metrics_dataframe['metric'].append('pgu std')
        metrics_dataframe['value'].append(method_results['pgu_std'])
        if 'fa_mean' in method_results:
            metrics_dataframe['dataset'].append(dataset)
            metrics_dataframe['method'].append(method)
            metrics_dataframe['dataset'].append(dataset)
            metrics_dataframe['method'].append(method)
            metrics_dataframe['dataset'].append(dataset)
            metrics_dataframe['metric'].append('fa')
            metrics_dataframe['value'].append(method_results['fa_mean'])
            metrics_dataframe['metric'].append('fa std')
            metrics_dataframe['value'].append(method_results['fa_std'])
            metrics_dataframe['metric'].append('fa interactions')
            metrics_dataframe['value'].append(method_results['fa_interactions_mean'])

In [ ]:
for k, v in metrics_dataframe.items():
    print(k, len(v))

In [ ]:
metrics_dataframe = pd.DataFrame(metrics_dataframe)

In [ ]:
metrics_dataframe

In [ ]:
metrics_dataframe = reshape_results_to_wide_format(metrics_dataframe)

In [ ]:
metrics_dataframe

In [ ]:
metrics_df_mean = metrics_dataframe[[
   #  'QM9 Simple Linear_fa', 'QM9 Simple Linear_pgi', 'QM9 Simple Linear_pgu',
   #  'QM9 Piecewise Linear_fa', 'QM9 Piecewise Linear_pgi', 'QM9 Piecewise Linear_pgu',
   # 'QM9 Nonlinear_fa', 'QM9 Nonlinear_pgi', 'QM9 Nonlinear_pgu',
    "hERG_pgi", "hERG_pgu",
    'COF_pgi', 'COF_pgu',

]]
metrics_df_std = metrics_dataframe[[
    # 'QM9 Simple Linear_fa std', 'QM9 Simple Linear_pgi std', 'QM9 Simple Linear_pgu std',
    # 'QM9 Piecewise Linear_fa std', 'QM9 Piecewise Linear_pgi std', 'QM9 Piecewise Linear_pgu std',
    # 'QM9 Nonlinear_fa std', 'QM9 Nonlinear_pgi std', 'QM9 Nonlinear_pgu std',
    "hERG_pgi std", "hERG_pgu std",
    'COF_pgi std', 'COF_pgu std',

]]
print(metrics_df_std)
metrics_df_std = metrics_df_std.rename(
    columns={
        'QM9 Simple Linear_pgi std': 'QM9 Simple Linear_pgi',
        'QM9 Simple Linear_pgu std': 'QM9 Simple Linear_pgu',
        'QM9 Piecewise Linear_pgi std': 'QM9 Piecewise Linear_pgi',
        'QM9 Piecewise Linear_pgu std': 'QM9 Piecewise Linear_pgu',
        'QM9 Nonlinear_pgi std': 'QM9 Nonlinear_pgi',
        'QM9 Nonlinear_pgu std': 'QM9 Nonlinear_pgu',
        'COF_pgi std': 'COF_pgi',
        'COF_pgu std': 'COF_pgu',
        'QM9 Simple Linear_fa std': 'QM9 Simple Linear_fa',
        'QM9 Piecewise Linear_fa std': 'QM9 Piecewise Linear_fa',
        'QM9 Nonlinear_fa std': 'QM9 Nonlinear_fa',
        "hERG_pgi std": "hERG_pgi",
        "hERG_pgu std": "hERG_pgu",
    }
)
df_mean_str = metrics_df_mean.map('{:.2f}'.format)
df_std_str = metrics_df_std.map('{:.2f}'.format)

# Concatenate the string DataFrames
# This works element-wise because the index and columns match perfectly
df_combined = df_mean_str + '(' + df_std_str + ')'
df_combined

In [ ]:
# metrics_dataframe = df_combined[[
#     'QM9 Simple Linear_pgi', 'QM9 Simple Linear_pgu',
#     'QM9 Piecewise Linear_pgi', 'QM9 Piecewise Linear_pgu',
#     'QM9 Nonlinear_pgi', 'QM9 Nonlinear_pgu',
#     'Batteries_pgi', 'Batteries_pgu',
# ]]
metrics_dataframe = df_combined.T
metrics_dataframe = metrics_dataframe.rename(
    columns={
        'lime': 'LIME',
        'shap': 'SHAP',
        'shapiq1': 'SHAP-IQ-1',
        'shapiq2': 'SHAP-IQ-2',
        'meg': 'MEG',
        'mmace': 'MMACE',
        'aggregated': 'Aggregated',
    }
)
metrics_dataframe = metrics_dataframe[[
    'LIME', 'SHAP', 'SHAP-IQ-1', 'SHAP-IQ-2', 'MMACE', 'MEG', 'Aggregated'
]]
print(metrics_dataframe.T.style.format(precision=3).to_latex())

In [ ]:
from src.analysis.xai_eval import rank_correlation
import numpy as np
from scripts.bulk_metric_compute_gt import aggregate_rankings_by_mean_position, convert_term_ranking_to_feature_ranking


import pandas as pd
import pickle

all_datasets = ['real_data/data_batteries_ecfp_descriptor', 'herg_data/data_herg_ecfp', 'synthetic_data/qm9_simple_linear6', 'synthetic_data/qm9_piecewise_linear_6','synthetic_data/qm9_nonlinear_6']
dataset_names = ['COF', 'hERG', 'QM9 Simple Linear', 'QM9 Piecewise Linear', 'QM9 Nonlinear']
# all_datasets = ['gt_synthetic_data/qm9_simple_linear6', 'gt_synthetic_data/qm9_piecewise_linear_6','gt_synthetic_data/qm9_nonlinear_6']
# dataset_names = ['QM9 Simple Linear', 'QM9 Piecewise Linear', 'QM9 Nonlinear']

correlations = {}

for i, dataset in enumerate(all_datasets):
    with open(f'../results/{dataset}/explanations/analysis/ranking_per_fold_results.pickle', 'rb') as f:
        results = pickle.load(f)
    agg_ranking_per_fold_dict = []
    for j in range(len(results['lime'])):
        ranking_list = [results[key][i] for key in results.keys()]
        ranking_list = [convert_term_ranking_to_feature_ranking(r) for r in ranking_list]
        _, aggregated_ranking = aggregate_rankings_by_mean_position(ranking_list)
        aggregated_ranking_df = pd.DataFrame({'features': aggregated_ranking.keys(), 'abs_ranking': aggregated_ranking.values()})
        aggregated_ranking_df['abs_ranking'] *= -1
        agg_ranking_per_fold_dict.append(aggregated_ranking_df)
    results['aggregated'] = agg_ranking_per_fold_dict

    pairs_of_ranks = [(key1, key2) for key1 in results.keys() for key2 in results.keys()]
    correla = {}
    for key1, key2 in pairs_of_ranks:
        corrs = []
        for ii in range(len(results[key1])):
            rank1 = convert_term_ranking_to_feature_ranking(results[key1][ii])
            rank2 = convert_term_ranking_to_feature_ranking(results[key2][ii])
            c = rank_correlation(rank1, rank2).statistic
            corrs.append(c)
        correlation = np.mean(corrs)
        correla[(key1, key2)] = correlation
        # print(f"{key1} vs {key2}: {correlation:.4f}")
        # print('--' * 20)
    print(i, dataset)
    correlations[dataset_names[i]] = correla


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# fig, ax = plt.subplots(1, len(all_datasets)+1, figsize=(14, 5), sharey=True, sharex=True, )

fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 6, figure=fig, wspace=0.1, hspace=0.55)

ax1 = fig.add_subplot(gs[0, 0:2])
ax2 = fig.add_subplot(gs[0, 2:4], sharey=ax1, sharex=ax1)
ax3 = fig.add_subplot(gs[0, 4:6], sharey=ax1, sharex=ax1)
ax4 = fig.add_subplot(gs[1, 1:3]) # Skip column 0
ax5 = fig.add_subplot(gs[1, 3:5], sharey=ax4, sharex=ax4)
ax = [ax1, ax2, ax3, ax4, ax5]

name_data_mapping = {
    'COF': 'COF',
    'QM9 Simple Linear': 'Linear',
    'QM9 Piecewise Linear': 'Piecewise Linear',
    'QM9 Nonlinear': 'Polynomial',
    'hERG': 'hERG'
}
order_mapping = {
    'COF': 4,
    'QM9 Simple Linear': 0,
    'QM9 Piecewise Linear': 1,
    'QM9 Nonlinear': 2,
    'hERG': 3
}

for i, (key, data) in enumerate(correlations.items()):
    if key not in name_data_mapping:
        continue
    s = pd.Series(data)
    correlation_matrix = s.unstack()
    name_mapping = {
        'lime': 'LIME',
        'shap': 'SHAP',
        'shapiq1': 'SHAP-IQ-1',
        'shapiq2': 'SHAP-IQ-2',
        'meg': 'MEG',
        'mmace': 'MMACE',
        'aggregated': 'Aggregated'
    }
    renamed_matrix = correlation_matrix.rename(index=name_mapping, columns=name_mapping)
    display(renamed_matrix)
    new_order = ['LIME', 'SHAP', 'SHAP-IQ-1', 'SHAP-IQ-2', 'MMACE', 'MEG', 'Aggregated']
    renamed_matrix = renamed_matrix[new_order].loc[new_order]
    mappable = sns.heatmap(renamed_matrix, annot=True, fmt=".1f", ax=ax[order_mapping[key]], cmap='coolwarm', vmin=-1, vmax=1, cbar=False, annot_kws={'size': 13})

    ax[order_mapping[key]].set_title(name_data_mapping[key], fontsize=16)
    ax[order_mapping[key]].tick_params(axis='both', which='major', labelsize=14)

for i, a in enumerate(ax):
    # We want labels on the first plot of each conceptual row (ax1 and ax4)
    if i in [0, 3]: # This corresponds to ax1 and ax4
        a.tick_params(labelleft=True)
    else: # This corresponds to ax2, ax3, ax5
        a.tick_params(labelleft=False)

# Also ensure the bottom plots have their x-tick labels
ax4.tick_params(labelbottom=True)
ax5.tick_params(labelbottom=True)

plt.tight_layout()
plt.savefig('../results/correlation_matrices_rf_herg.pdf', dpi=300, bbox_inches='tight')
plt.show()